# Statistical Tests

Tests whether safety scores are normally distributed within each census region (Shapiro-Wilk),
whether regional medians differ significantly (Kruskal-Wallis), and whether similar counties
cluster geographically (Global Moran's I with LISA quadrant breakdown).

**Input:** `data/processed/final_data.csv`  
**Output:** printed test results

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.neighbors import NearestNeighbors

C:\Users\Josh\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Josh\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
processed_data_path = "../data/processed/"
df = pd.read_csv(f"{processed_data_path}final_data.csv")
df.head()


,state_fips,county_fips,poverty_rate,median_hhi,name,state_abbr,fips,ACCESS2,ARTHRITIS,BINGE,...,traffic_fatality_rate,traffic_score,air_quality_score,health_score,economic_score,lat,lon,safety_score,KM_cluster,HDB_cluster
0,1,1,11.7,68857.0,Autauga County,AL,1001,10.4,28.2,15.5,...,18.614096,65.265949,45.801081,50.141087,54.810415,32.532237,-86.646439,54.527409,1,0
1,1,3,10.0,74248.0,Baldwin County,AL,1003,9.5,26.4,17.8,...,9.193712,72.903972,47.368421,53.322841,58.504260,30.659218,-87.746067,58.744859,1,0
2,1,5,25.5,45298.0,Barbour County,AL,1005,17.2,30.6,13.4,...,44.063451,55.558230,45.423513,37.220930,31.788070,31.870253,-85.405103,43.686194,0,0
3,1,7,19.4,56025.0,Bibb County,AL,1007,14.3,29.5,15.4,...,22.244962,63.283847,45.377228,43.124236,42.072571,33.015893,-87.127148,49.525034,0,0
4,1,9,12.8,64962.0,Blount County,AL,1009,13.1,28.4,16.0,...,30.487288,59.741928,44.027698,46.209468,52.280110,33.977357,-86.566440,50.937892,0,0


In [3]:
print("Shapiro-Wilk normality test per region:")
for region, group in df.groupby('census_region'):
    sample = group['safety_score'].dropna()
    if len(sample) > 500:
        sample = sample.sample(500, random_state=42)
    stat, p = stats.shapiro(sample)
    print(f"  {region}: W={stat:.4f}, p={p:.4f}")


Shapiro-Wilk normality test per region:
  Midwest: W=0.9751, p=0.0000
  Northeast: W=0.9696, p=0.0001
  South: W=0.9643, p=0.0000
  West: W=0.9827, p=0.0000


In [4]:
groups = [group['safety_score'].dropna().values for _, group in df.groupby('census_region')]
stat, p = stats.kruskal(*groups)
print(f"\nKruskal-Wallis: H={stat:.4f}, p={p:.4f}")

print("\nMedian safety_score by region:")
print(df.groupby('census_region')['safety_score'].median().sort_values())



Kruskal-Wallis: H=700.4867, p=0.0000

Median safety_score by region:
census_region
South        51.396024
Midwest      56.487966
West         58.880761
Northeast    59.759736
Name: safety_score, dtype: float64


## Global Moran's I: Spatial Autocorrelation

Tests whether similar counties cluster geographically (positive I) or are dispersed (negative I).

- **I = 1**: perfect clustering
- **I approx 0**: random spatial arrangement
- **I = -1**: perfect dispersion

Uses k=8 nearest county centroids with row-standardised weights.

In [5]:
valid = df.dropna(subset=['lat', 'lon', 'safety_score']).copy()
coords = valid[['lat', 'lon']].values

nbrs = NearestNeighbors(n_neighbors=9, algorithm='ball_tree').fit(coords)
_, indices = nbrs.kneighbors(coords)
neighbor_idx = indices[:, 1:]  # drop self-reference

x = valid['safety_score'].values
x_d = x - x.mean()
spatial_lag = np.mean(x_d[neighbor_idx], axis=1)  # row-standardised lag
I = float(np.sum(x_d * spatial_lag) / np.sum(x_d**2))
EI = -1.0 / (len(x) - 1)
z_approx = (I - EI) / (1.0 / np.sqrt(len(x)))

print(f"Global Moran's I = {I:.4f}")
print(f"Expected I (random) = {EI:.6f}")
print(f"Z-score (approx) = {z_approx:.2f}")
print(f"Interpretation: {'Strong' if I > 0.5 else 'Moderate' if I > 0.2 else 'Weak'} positive spatial autocorrelation")

Global Moran's I = 0.5254
Expected I (random) = -0.000318
Z-score (approx) = 29.48
Interpretation: Strong positive spatial autocorrelation


In [6]:
# LISA quadrant breakdown
x_std = (x - x.mean()) / x.std()
lag_std = (spatial_lag - spatial_lag.mean()) / spatial_lag.std()

quadrants = {
    'HH (safe surrounded by safe)':     int(((x_std >= 0) & (lag_std >= 0)).sum()),
    'LL (unsafe surrounded by unsafe)':  int(((x_std <  0) & (lag_std <  0)).sum()),
    'HL (safe island in danger region)': int(((x_std >= 0) & (lag_std <  0)).sum()),
    'LH (danger pocket in safe region)': int(((x_std <  0) & (lag_std >= 0)).sum()),
}
for label, count in quadrants.items():
    print(f"  {label}: {count}")

  HH (safe surrounded by safe): 1123
  LL (unsafe surrounded by unsafe): 1270
  HL (safe island in danger region): 337
  LH (danger pocket in safe region): 414


### Interpretation

**Moran's I ≈ 0.53** (p < 0.001) confirms strong positive spatial autocorrelation:
safe counties cluster together geographically, and so do unsafe ones.

The breakdown shows most counties are in HH or LL quadrants. HL and LH outliers are the most
analytically interesting: counties that deviate sharply from their neighbours.